# Resnet

In [ ]:
import torchvision
from torch import nn
import torch

## Resnet

[resnet paper](https://arxiv.org/pdf/1512.03385.pdf)

solves the problem of vanishing gradient by providing a skip connection where the gradients can flow directly from the n<sub>n-1</sub> to n<sub>n-2</sub> layer by having a skip connection

<img src=https://neurohive.io/wp-content/uploads/2019/01/resnet-e1548261477164.png, width=400/>

<img src=https://neurohive.io/wp-content/uploads/2019/01/resnet-architectures-34-101.png, width=700/>

## 34 layer CNN model

In [ ]:
def conv_layer(c_in, c_out, ks, stride):
    return nn.Sequential(nn.Conv2d(c_in, c_out, kernel_size=ks, padding=ks//2, stride=stride),
                         nn.BatchNorm2d(c_out),
                         nn.ReLU())

In [ ]:
model = nn.Sequential(conv_layer(3, 32, 7, 2),
              nn.Sequential(*[conv_layer(32 if i==0 else 64, 64, ks=3, stride=2 if i==0 else 1) for i in range(6)]),
              nn.Sequential(*[conv_layer(64 if i==0 else 128, 128, ks=3, stride=2 if i==0 else 1) for i in range(8)]),
              nn.Sequential(*[conv_layer(128 if i==0 else 256, 256, ks=3, stride=2 if i==0 else 1) for i in range(12)]),
              nn.Sequential(*[conv_layer(256 if i==0 else 512, 512, ks=3, stride=2 if i==0 else 1) for i in  range(6)]),
              nn.AvgPool2d(4),
              nn.Flatten(),
              nn.Linear(512, 1000))

In [ ]:
model(torch.randn(2, 3, 128, 128)).shape

## Resnet model architecture

In [ ]:
class block(nn.Module):
    def __init__(
        self, in_channels, intermediate_channels, identity_downsample=None, stride=1
    ):
        super(block, self).__init__()
        self.expansion = 4
        self.conv1 = nn.Conv2d(
            in_channels, intermediate_channels, kernel_size=1, stride=1, padding=0, bias=False
        )
        self.bn1 = nn.BatchNorm2d(intermediate_channels)
        self.conv2 = nn.Conv2d(
            intermediate_channels,
            intermediate_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False
        )
        self.bn2 = nn.BatchNorm2d(intermediate_channels)
        self.conv3 = nn.Conv2d(
            intermediate_channels,
            intermediate_channels * self.expansion,
            kernel_size=1,
            stride=1,
            padding=0,
            bias=False
        )
        self.bn3 = nn.BatchNorm2d(intermediate_channels * self.expansion)
        self.relu = nn.ReLU()
        self.identity_downsample = identity_downsample
        self.stride = stride

    def forward(self, x):
        identity = x.clone()

        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.conv3(x)
        x = self.bn3(x)

        if self.identity_downsample is not None:
            identity = self.identity_downsample(identity)

        x += identity
        x = self.relu(x)
        return x

<img src=https://neurohive.io/wp-content/uploads/2019/01/resnet-architecture-3.png width=400/>

In [ ]:
class ResNet(nn.Module):
    def __init__(self, block, layers, image_channels, num_classes):
        super(ResNet, self).__init__()
        self.in_channels = 64
        self.conv1 = nn.Conv2d(image_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU()
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        # Essentially the entire ResNet architecture are in these 4 lines below
        self.layer1 = self._make_layer(
            block, layers[0], intermediate_channels=64, stride=1
        )
        self.layer2 = self._make_layer(
            block, layers[1], intermediate_channels=128, stride=2
        )
        self.layer3 = self._make_layer(
            block, layers[2], intermediate_channels=256, stride=2
        )
        self.layer4 = self._make_layer(
            block, layers[3], intermediate_channels=512, stride=2
        )

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * 4, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = x.reshape(x.shape[0], -1)
        x = self.fc(x)

        return x

    def _make_layer(self, block, num_residual_blocks, intermediate_channels, stride):
        identity_downsample = None
        layers = []

        if stride != 1 or self.in_channels != intermediate_channels * 4:
            identity_downsample = nn.Sequential(
                nn.Conv2d(
                    self.in_channels,
                    intermediate_channels * 4,
                    kernel_size=1,
                    stride=stride,
                    bias=False
                ),
                nn.BatchNorm2d(intermediate_channels * 4),
            )

        layers.append(
            block(self.in_channels, intermediate_channels, identity_downsample, stride)
        )

        self.in_channels = intermediate_channels * 4

        for i in range(num_residual_blocks - 1):
            layers.append(block(self.in_channels, intermediate_channels))

        return nn.Sequential(*layers)

Now lets define a resnet model according to the block size defined in the paper

In [ ]:
def ResNet50(img_channel=3, num_classes=1000):
    return ResNet(block, [3, 4, 6, 3], img_channel, num_classes)


def ResNet101(img_channel=3, num_classes=1000):
    return ResNet(block, [3, 4, 23, 3], img_channel, num_classes)


def ResNet152(img_channel=3, num_classes=1000):
    return ResNet(block, [3, 8, 36, 3], img_channel, num_classes)

In [ ]:
ResNet50()